# Phase 1 / Gate 1 — jWave reference-simulation reproduction

**Protocol:** `acoustic_simulation_phase_protocol.md`, Phase 1
**Purpose:** reproduce jWave's own documented "Homogeneous Medium" example
(https://ucl-bug.github.io/jwave/notebooks/ivp/homogeneous_medium.html) on
a GPU runtime, exactly as published, before trusting the toolkit on
anything else. This is the toolkit-reference check required before Gate 1
can pass — do not skip it.

**Run this in Google Colab** with a GPU runtime (Runtime → Change runtime
type → GPU). Record: jax/jwave versions, GPU model, the timing result, and
whether the pressure field plot shows a circular wavefront expanding at the
correct speed (1500 m/s, i.e. traveling `dx * N_steps_of_travel` per the
known CFL-scaled timestep — visually: a clean expanding ring, no
checkerboarding/blow-up/asymmetry).

After running, report back: versions, GPU, timing, and whether the
wavefield looked physically correct — that gets logged in `jwave/LOG.md`
as the Gate 1 run entry.

In [ ]:
# Install jWave (pulls in a compatible jax/jaxlib automatically).
# Colab ships a GPU-enabled jax already; this just adds jwave on top.
!pip install -q jwave

In [ ]:
# jwave's install pulls jax's core package down to an older version
# (0.4.38) to satisfy its dependency pin, but leaves Colab's newer
# preinstalled CUDA-plugin libraries (jaxlib/jax-cuda12-plugin) in place --
# an old jax core calling a newer CUDA plugin's internal API causes
# "AttributeError: module 'jax._src.lib.triton' has no attribute
# 'register_compilation_handler'". Fix: force-reinstall a matching
# jax+jaxlib+cuda12-plugin set at the SAME version.
!pip install -q -U "jax[cuda12]==0.4.38"
print("Reinstalled matching jax[cuda12]==0.4.38 -- RESTART THE RUNTIME NOW "
      "(Runtime > Restart session), then re-run all cells from the top.")

In [ ]:
import jax
import jwave
import importlib.metadata

print("jax version:", jax.__version__)
# jwave doesn't expose __version__ directly -- read it from package metadata.
print("jwave version:", importlib.metadata.version("jwave"))
print("jax devices:", jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU device visible — check Runtime > Change runtime type."

In [ ]:
from jax import numpy as jnp
from jax import jit
from matplotlib import pyplot as plt

from jwave import FourierSeries
from jwave.geometry import Domain, Medium, TimeAxis, circ_mask
from jwave.acoustics import simulate_wave_propagation
from jwave.utils import show_field

# Domain setup — exactly as in jWave's published example.
N, dx = (128, 128), (0.1e-3, 0.1e-3)
domain = Domain(N, dx)

# Acoustic medium: constant 1500 m/s (water-like), homogeneous.
medium = Medium(domain=domain, sound_speed=1500.0)

# CFL = 0.3 — stability condition from the published example.
time_axis = TimeAxis.from_medium(medium, cfl=0.3)

# Initial pressure: circular source, radius 4 grid points at (80, 60).
p0 = 1.0 * jnp.expand_dims(circ_mask(N, 4, (80, 60)), -1)
p0 = FourierSeries(p0, domain)

show_field(p0)
plt.title("Initial pressure field")
plt.show()

In [ ]:
@jit
def compiled_simulator(medium, p0):
    return simulate_wave_propagation(medium, time_axis, p0=p0)

# Warm-up call to trigger JIT compilation (excluded from timing).
pressure = compiled_simulator(medium, p0)
pressure.block_until_ready() if hasattr(pressure, "block_until_ready") else None

t = 250
show_field(pressure[t])
plt.title(f"Pressure field at t={time_axis.to_array()[t]}")
plt.show()

In [ ]:
# Timed run (post-JIT) — this is the single-simulation timing Gate 1 needs
# for the Phase 4 compute-budget estimate later.
%timeit compiled_simulator(medium, p0)

## Gate 1 checklist (fill in after running)

- [ ] jax/jwave versions recorded above
- [ ] GPU model recorded (`!nvidia-smi`)
- [ ] Pressure field at t=250 shows a clean expanding circular wavefront
      (no divergence, no checkerboard artifact, roughly circular/symmetric)
- [ ] Per-simulation timing recorded (`%timeit` output above)

Report these back so they can be logged in `jwave/LOG.md` as the Gate 1 run,
and versions pinned into `jwave/requirements.txt`.

In [ ]:
!nvidia-smi